# 03 · Data Preparation

**CRISP-DM phase 3** — apply the cleaning decisions and produce the analysis-ready dataset.

All logic lives in `src/data.py::clean()` (one place, reused by every notebook, with `assert` checks on its output).
This notebook **shows the effect of each decision** (before → after) and the reason for it.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / "src"))
from data import ERROR_CODES, SENSORS, clean, load_raw, save_clean

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)

raw = load_raw()
df = clean(raw)
print(f"raw: {raw.shape}  ->  clean: {df.shape}")

raw: (259200, 16)  ->  clean: (253856, 15)


## DQ-01 · Timestamps → one `date` column
Three formats parsed **explicitly** (no automatic guessing): ISO, `dd/mm/yyyy`, `Mon d, yyyy`.
Two of them carry no time of day, so the common grain is the **day**.

In [2]:
examples = raw.groupby(raw["event_timestamp"].str.replace(r"\d", "9", regex=True)).head(1)[["batch_id", "event_timestamp"]]
examples.merge(df[["batch_id", "date"]], on="batch_id")

,batch_id,event_timestamp,date
0,BATCH-2023-000001,2023-06-02 03:05:00,2023-06-02
1,BATCH-2023-000011,16/06/2023,2023-06-16
2,BATCH-2023-000022,"May 15, 2023",2023-05-15
3,BATCH-2023-000077,"Jun 09, 2023",2023-06-09
4,BATCH-2023-000169,"Apr 08, 2023",2023-04-08


In [3]:
print("Unparsed dates:", df["date"].isna().sum())
print("Range:", df["date"].min().date(), "->", df["date"].max().date(), "| days:", df["date"].nunique())

Unparsed dates: 0
Range: 2023-04-01 -> 2023-06-29 | days: 90


## DQ-02 · Fahrenheit → Celsius
`temp_value` + `temp_unit` become a single `reaction_temperature_celsius` (the documented name). °C = (°F − 32) × 5/9.

In [4]:
f_ids = raw.loc[raw["temp_unit"] == "F", "batch_id"]
before = raw.loc[raw["batch_id"].isin(f_ids)].set_index("batch_id")[["temp_value", "temp_unit"]]
after = df.loc[df["batch_id"].isin(f_ids)].set_index("batch_id")["reaction_temperature_celsius"]
before.join(after).head()

,temp_value,temp_unit,reaction_temperature_celsius
batch_id,,,
BATCH-2023-000481,186.83258,F,86.0181
BATCH-2023-001533,177.28268,F,80.7126
BATCH-2023-001680,178.58462,F,81.4359
BATCH-2023-001777,191.34572,F,88.5254
BATCH-2023-001984,194.85104,F,90.4728


## DQ-03 · Error codes → missing
Values −1 bar, 860 g/cm³, pH 14 and 210 °C (= 410 °F) are not measurements: they are set to `NaN`.
A flag `had_error_code` keeps track of affected batches (useful to check hypothesis H4 later).

In [5]:
counts = pd.DataFrame({
    "before (raw)": [(raw[c if c != "reaction_temperature_celsius" else "temp_value"].isin([v, 410.0] if v == 210 else [v])).sum()
                     for c, v in ERROR_CODES.items()],
    "after (clean)": [np.isclose(df[c], v).sum() for c, v in ERROR_CODES.items()],
}, index=[f"{c} = {v}" for c, v in ERROR_CODES.items()])
counts

,before (raw),after (clean)
reaction_temperature_celsius = 210.0,526,0
vessel_pressure_bar = -1.0,544,0
density_g_cm3 = 860.0,558,0
ph_level = 14.0,543,0


In [6]:
print("Batches flagged had_error_code:", df["had_error_code"].sum())
df.groupby("plant", observed=True)["had_error_code"].sum()

Batches flagged had_error_code: 2136


plant
Plant-01       0
Plant-02       0
Plant-03       0
Plant-04    2136
Plant-05       0
Name: had_error_code, dtype: int64

## DQ-03b · Marginal excursions → kept
Continuous values just outside the normal ranges are plausible real readings → **no action**.
Removing them would hide exactly the kind of process deviation we are looking for.

## DQ-04 · Missing values → kept as NaN
Missingness is MCAR-like (phase 2), so analyses on available data are unbiased.
No global imputation here: imputing is a **modelling decision** (done inside each model, fitted on training data only, to avoid leakage).

In [7]:
pd.DataFrame({
    "% missing raw": raw[["temp_value"] + SENSORS[1:]].isna().mean().values * 100,
    "% missing clean": df[SENSORS].isna().mean().values * 100,
}, index=SENSORS).round(2)

,% missing raw,% missing clean
reaction_temperature_celsius,8.94,7.23
vessel_pressure_bar,8.86,7.16
refractive_index,8.86,6.94
density_g_cm3,8.89,7.19
ph_level,8.85,7.14
mixing_torque_nm,8.83,6.91
mass_flow_rate_kg_h,8.94,7.02


Missing goes **down** (~8.9% → ~7%) because DQ-05 removes the fully empty rows; converting error codes to NaN adds a little back for 4 sensors (≈0.2 points).

In [8]:
complete = df[SENSORS].notna().all(axis=1)
print(f"Rows with all 7 sensors present: {complete.sum():,} ({complete.mean():.1%})")
print("-> dropping every incomplete row would lose ~40% of the data: another reason not to do it.")

Rows with all 7 sensors present: 151,958 (59.9%)
-> dropping every incomplete row would lose ~40% of the data: another reason not to do it.


## DQ-05 · Batches with no sensor data → dropped
5,344 batches have all 7 sensors empty: they carry no process information. Their stability is the same as the rest (phase 2),
so dropping them (~2% of rows) does not bias the data.

In [9]:
all_empty = raw[["temp_value"] + SENSORS[1:]].isna().all(axis=1)
print("Dropped rows:", all_empty.sum(), f"({all_empty.mean():.1%})")
print("Mean stability | dropped:", raw.loc[all_empty, "stability_months"].mean().round(3),
      "| kept:", raw.loc[~all_empty, "stability_months"].mean().round(3))

Dropped rows: 5344 (2.1%)
Mean stability | dropped: 21.497 | kept: 21.532


## DQ-06 / DQ-07 / DQ-08 · Columns
- **DQ-06**: `plant_country` dropped (1:1 with plant); `plant_location` renamed `plant`.
- **DQ-07**: `purity_grade` and `oxidation_risk_flag` kept **only for business reporting**; they are listed as leakage columns in `config.yaml` and never used as features.
- **DQ-08**: new `reactor_id` = plant × reactor (e.g. `P04-R03`) → 37 physical reactors.

In [10]:
print("Columns:", list(df.columns))
print("Physical reactors:", df["reactor_id"].nunique())
df.head()

Columns: ['batch_id', 'date', 'plant', 'reactor_id', 'reaction_temperature_celsius', 'vessel_pressure_bar', 'refractive_index', 'density_g_cm3', 'ph_level', 'mixing_torque_nm', 'mass_flow_rate_kg_h', 'had_error_code', 'stability_months', 'purity_grade', 'oxidation_risk_flag']
Physical reactors: 37


,batch_id,date,plant,reactor_id,reaction_temperature_celsius,vessel_pressure_bar,refractive_index,density_g_cm3,ph_level,mixing_torque_nm,mass_flow_rate_kg_h,had_error_code,stability_months,purity_grade,oxidation_risk_flag
0,BATCH-2023-000001,2023-06-02,Plant-05,P05-R02,82.3319,2.46575,1.465316,0.858032,NaN,11.97486,492.9606,False,21.090538,Industrial Grade,False
1,BATCH-2023-000002,2023-04-12,Plant-05,P05-R04,91.7391,2.24107,1.455938,0.876435,5.79598,11.85726,449.9237,False,21.202418,Industrial Grade,False
2,BATCH-2023-000003,2023-06-03,Plant-03,P03-R07,85.0281,2.34404,1.468787,0.848555,5.31299,12.02680,437.1050,False,22.515180,Industrial Grade,False
3,BATCH-2023-000004,2023-05-11,Plant-04,P04-R05,88.5994,2.10548,1.447880,0.871153,5.44103,11.67030,439.3516,False,19.843445,Technical Grade,True
4,BATCH-2023-000005,2023-05-19,Plant-01,P01-R01,82.4168,2.16206,NaN,0.840569,5.06367,12.24158,447.2072,False,24.464046,Industrial Grade,False


## Final dataset

In [11]:
df.dtypes

batch_id                        string[python]
date                            datetime64[ns]
plant                                 category
reactor_id                            category
reaction_temperature_celsius           float64
vessel_pressure_bar                    float64
refractive_index                       float64
density_g_cm3                          float64
ph_level                               float64
mixing_torque_nm                       float64
mass_flow_rate_kg_h                    float64
had_error_code                            bool
stability_months                       float64
purity_grade                          category
oxidation_risk_flag                    boolean
dtype: object

In [12]:
df.describe().T.round(3)

,count,mean,min,25%,50%,75%,max,std
date,253856,2023-05-15 13:51:25.503592704,2023-04-01 00:00:00,2023-04-23 00:00:00,2023-05-16 00:00:00,2023-06-07 00:00:00,2023-06-29 00:00:00,NaN
reaction_temperature_celsius,235504.0,85.953523,78.9556,82.7733,85.94415,89.141425,93.1264,3.695834
vessel_pressure_bar,235681.0,2.307963,1.98931,2.1608,2.30789,2.45481,2.6196,0.170178
refractive_index,236229.0,1.46066,1.437219,1.450678,1.460645,1.470644,1.483709,0.011674
density_g_cm3,235599.0,0.860023,0.837224,0.849978,0.860032,0.870086,0.885914,0.011712
ph_level,235720.0,5.503103,4.94506,5.254157,5.501595,5.753275,6.111014,0.29103
mixing_torque_nm,236320.0,11.941048,10.035246,11.7099,11.97424,12.23888,12.56448,0.392478
mass_flow_rate_kg_h,236038.0,453.558692,383.2505,421.858325,453.3501,485.355525,521.9248,36.897379
stability_months,253856.0,21.53232,18.01519,19.911692,21.229628,22.913003,28.609162,2.045567


In [13]:
# Sanity check: the main phase-2 finding is unchanged after cleaning
AFFECTED = ["P04-R03", "P04-R07", "P04-R08"]
period = np.where(df["date"] >= "2023-05-08", "from 2023-05-08", "before")
group = np.where(df["reactor_id"].isin(AFFECTED), "3 affected reactors", "other 34 reactors")
df.pivot_table(index=group, columns=period, values="stability_months", aggfunc="mean").round(2)[["before", "from 2023-05-08"]]

,before,from 2023-05-08
3 affected reactors,21.85,18.61
other 34 reactors,21.68,21.68


In [14]:
path = save_clean(df)
print("Saved:", path.relative_to(Path.cwd().parent), f"({path.stat().st_size / 1e6:.1f} MB)")

Saved: data\processed\fragrance_clean.parquet (12.4 MB)
